## Week 2 Day 2 — Groq model + Resend email

A community contribution combining two changes to the original Lab 2:
1. **Model:** instead of OpenAI's `gpt-4o-mini`, we use **Groq** through its OpenAI-compatible API (no OpenAI quota needed).
2. **Email:** instead of SendGrid, we send the cold sales email through the **Resend** API.

Everything else is the same agentic SDR project from the lecture:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

## Before we start - some setup:

### 1. Groq (the model provider)

Sign up for a free key at https://console.groq.com/ , then add it to your `.env`:

`GROQ_API_KEY=xxxx`

### 2. Resend (for sending email)

Sign up for a free account at https://resend.com/ . Create an API key (API Keys >> Create API Key) and add it to your `.env`:

`RESEND_API_KEY=xxxx`

Without a verified domain, Resend lets you send **from** the test address `onboarding@resend.dev`, and you can only send **to** the email address you registered your Resend account with. Put that address in your `.env` as well:

`GMAIL_TO=your_resend_account_email@example.com`

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
import requests
import asyncio

In [2]:
load_dotenv(override=True)

# No OpenAI quota, so use Groq through its OpenAI-compatible API (same as Lab 1).
# Note: llama-3.3-70b sometimes hallucinates tool NAMES in multi-tool/handoff flows,
# which Groq rejects with a hard 400 (tool_use_failed). gpt-oss-120b is far more
# reliable at tool calling, so we use it here.
groq_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)
groq_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

# Tracing uploads to platform.openai.com and needs an OpenAI key — disable it
set_tracing_disabled(True)

In [3]:
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
to_email = os.getenv("GMAIL_TO")
print("Sending test emails to:", to_email)

Sending test emails to: izmaanwar005@gmail.com


In [5]:
# Quick sanity check that the Resend key works before we wire it into agents.

def send_via_resend(subject: str, html_body: str) -> Dict[str, str]:
    """ Send an email through the Resend API and return a status dict """
    from_email = "onboarding@resend.dev"  # Resend's test sender (no domain verification needed)

    headers = {
        "Authorization": f"Bearer {RESEND_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "from": f"ComplAI Sales <{from_email}>",
        "to": [to_email],
        "subject": subject,
        "html": html_body,
    }

    response = requests.post("https://api.resend.com/emails", json=payload, headers=headers)

    # Resend returns 200 on success (SendGrid uses 202)
    if response.ok:
        return {"status": "success"}
    return {"status": "failure", "message": response.text}


print(send_via_resend("Resend test", "<p>This is a test email from the Groq + Resend lab.</p>"))

{'status': 'success'}


## Step 1: Agent workflow

In [6]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [7]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=groq_model
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=groq_model
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=groq_model
)

In [8]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

**Subject:** Accelerate Your SOC 2 Journey with AI‑Powered Automation  

---

Hi **[First Name]**,  

I’m reaching out from **ComplAI** because I see many fast‑growing companies like **[Company]** struggling to keep SOC 2 controls up‑to‑date while juggling product releases and day‑to‑day operations.  

**The challenge**  
- Manual evidence collection spreads teams thin.  
- Tracking control changes across multiple tools leads to gaps and missed deadlines.  
- Audit preparation often becomes a last‑minute sprint, increasing risk and cost.

**Our solution** – **ComplAI SOC 2 Automation Platform**  
- **AI‑driven evidence gathering** pulls logs, policies, and configuration data automatically from your existing stack.  
- **Real‑time control monitoring** flags drift before it becomes a compliance issue.  
- **Audit-ready reports** are generated with a single click, cutting prep time by up to 70 %.  
- Seamless integration with cloud providers, CI/CD pipelines, and ticketing systems—no heav

In [9]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

**Subject:** Accelerate Your SOC 2 Compliance with AI‑Powered Automation  

Hi **[First Name]**,

I’m reaching out from **ComplAI** – the AI‑driven platform that helps organizations like **[Company]** achieve and maintain SOC 2 compliance faster, cheaper, and with far fewer manual errors.

### Why most teams struggle with SOC 2  
- **Fragmented evidence collection** – data lives in dozens of tools, making audit prep a nightly nightmare.  
- **Resource‑intensive gap remediation** – identifying, assigning, and tracking controls can consume weeks of engineering time.  
- **Unpredictable audit timelines** – last‑minute evidence requests often delay audit completion and increase costs.

### How ComplAI solves it  
| Feature | Benefit |
|---|---|
| **Continuous Control Monitoring** | Real‑time visibility into all Trust Services Criteria (TSC) across your cloud, SaaS, and on‑prem environments. |
| **AI‑Generated Evidence** | Automated collection, classification, and tagging of logs, configs, 

In [10]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=groq_model
)

In [11]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
**Subject:** Accelerate Your SOC 2 Readiness with AI‑Powered Automation  

Hi {{FirstName}},  

I’m [Your Name] at **ComplAI**—the AI‑driven platform that removes the guesswork from SOC 2 compliance.  

- **Automated evidence collection** → up to 50% less manual work  
- **Continuous control monitoring** → real‑time readiness alerts  
- **Audit‑ready reports** → generate and export in minutes  

Teams that switch to ComplAI cut their audit preparation time from weeks to days and reduce compliance costs by up to 30%.  

Can we schedule a 15‑minute call next week to show how it works for companies like **[Similar Company/Industry]**?  

Best,  
[Your Name]  
Senior Sales Executive, ComplAI  
[Phone] | [email] | www.complai.com  

*P.S. I’ve attached a one‑pager that outlines the ROI you can expect.*


## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [12]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=groq_model,
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=groq_model,
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=groq_model,
)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [13]:
@function_tool
def send_email(body: str) -> Dict[str, str]:
    """ Send out an email with the given body to all sales prospects via Resend """
    from_email = "onboarding@resend.dev"  # Resend's test sender

    headers = {
        "Authorization": f"Bearer {RESEND_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "from": f"ComplAI Sales <{from_email}>",
        "to": [to_email],
        "subject": "Sales email",
        "html": f"<p>{body}</p>",
    }

    response = requests.post("https://api.resend.com/emails", json=payload, headers=headers)

    if response.ok:
        return {"status": "success"}
    return {"status": "failure", "message": response.text}

In [14]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects via Resend', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001B4E1009EE0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### And you can also convert an Agent into a tool

In [15]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001B4DF03DEE0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001B4DF03EA20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', descr

## And now it's time for our Sales Manager - our planning agent

In [16]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=groq_model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

In [ ]:
print(result)

### Didn't get an email?

First, check your Spam folder. Then `print(result)` and look for errors. With Resend, the two most common issues are:
- The `to` address must be the **same email you registered your Resend account with** (until you verify your own domain).
- The `from` address must stay as `onboarding@resend.dev` until you verify a domain.

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across

In [17]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=groq_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=groq_model)
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [18]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects using Resend """
    from_email = "onboarding@resend.dev"  # Resend's test sender

    headers = {
        "Authorization": f"Bearer {RESEND_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "from": f"ComplAI Sales <{from_email}>",
        "to": [to_email],
        "subject": subject,
        "html": html_body,
    }

    response = requests.post("https://api.resend.com/emails", json=payload, headers=headers)

    if response.ok:
        return {"status": "success"}
    return {"status": "failure", "message": response.text}

In [19]:
tools = [subject_tool, html_tool, send_html_email]
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001B4DF03C400>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001B4DEEA5B20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=No

In [20]:
instructions = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=groq_model,
    handoff_description="Convert an email to HTML and send it")

### Now we have 3 tools and 1 handoff

In [21]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001B4DF03DEE0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001B4DF03EA20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', descrip

In [22]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=groq_model)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

In [ ]:
print(result)

### Then check your email!!